# Оценка самоповторяемости на некоторых RUN'ах экспериментов

In [1]:
import sys
import os
import json

sys.path.append('./../')

In [2]:
import numpy as np
import pandas as pd
from dotenv import load_dotenv

from eval.self_repeatness import count_self_repeating_part
from eval.variarity import count_fields_variance
from eval.md_classes import MDField, DataStep

load_dotenv()

True

In [3]:
MODEL = 'gpt-5.4-nano'
MODEL_PROVIDER = 'openai'
TEMPERATURE = 0.0
COMPARER_SP = './../prompts/llm_judges/fields_comparer.txt'

STATS_DIR = './../data/valid_stats/'

DF_PATHS = {
    'm1': './../data/md_generations/m1_synth_bio_mds.csv',
    'm2': './../data/md_generations/m2_synth_bio_mds.csv',
    'm3': './../data/md_generations/m3_synth_bio_mds/m3_synth_bio_mds.csv',
    'm4': './../data/md_generations/m4_synth_bio_mds/m4_synth_bio_mds.csv',
}

EMBEDS_PATHS = {
    'm1': './../data/md_generations/embeds/m1/',
    'm2': './../data/md_generations/embeds/m2/',
    'm3': './../data/md_generations/embeds/m3_full/',
    'm4': './../data/md_generations/embeds/m4_full/',
}

QUERIES_CNT = 8
RUNS = 3

## Просто доп подгрузки для подсчета

In [4]:
from langchain_core.callbacks import BaseCallbackHandler


class TokenCounter(BaseCallbackHandler):
    def __init__(self):
        self.prompt_tokens     = 0
        self.completion_tokens = 0
        self.total_tokens      = 0
        self.n_calls           = 0

    def on_llm_end(self, response, **kwargs):
        usage = None

        # LangChain wraps different providers differently
        if hasattr(response, "llm_output") and response.llm_output:
            usage = response.llm_output.get("token_usage") or \
                    response.llm_output.get("usage")

        # OpenAI via langchain_openai exposes it here
        if usage is None and response.generations:
            for gen_list in response.generations:
                for gen in gen_list:
                    info = getattr(gen, "generation_info", None) or {}
                    usage = info.get("usage") or info.get("token_usage")
                    if usage:
                        break

        if usage:
            self.prompt_tokens     += usage.get("prompt_tokens",     0)
            self.completion_tokens += usage.get("completion_tokens", 0)
            self.total_tokens      += usage.get("total_tokens",      0)
        self.n_calls += 1

In [5]:
from typing import List

import time
from eval.judges.repeating.fields_repeating import CompareState, FieldsComparator

def read_prompts(pr_path:str) -> str:
    with open(pr_path, 'rt') as fio:
        res = fio.read()
    return res 

field_comparer = FieldsComparator(read_prompts(COMPARER_SP), MODEL, MODEL_PROVIDER, TEMPERATURE)

def factory_func(md_name: str, md_description: str):
    def llm_judge(target_field: str, candidates: List[str]) -> List[str]:
        is_complete = False

        while not is_complete:
            try:
                counter = TokenCounter()

                state = CompareState()
                state['md_schema_name'] = md_name
                state['md_schema_descr'] = md_description
                state['main_field_str'] = target_field
                state['potention_similar_fields'] = candidates
                
                wf = field_comparer.workflow
                res = wf.invoke(
                    state,
                    config={"callbacks": [counter]})
                print('Spend during last LLM call: ', str({
                    'prompt': counter.prompt_tokens,
                    'completion': counter.completion_tokens,
                    'total': counter.total_tokens
                }))

                if res['result'] != None and res['result'].same_fields_from_list != None:
                    is_complete = True
                print(res['result'].same_fields_from_list)
            except Exception as e:
                print(f'Catched \"{str(e)}\". Retrying in 120 sec')
                time.sleep(120)

        return res['result'].same_fields_from_list
    
    return llm_judge

## Запуск кода обхода таблицы и сохранения

### Проба сравнения

In [6]:
def field_converter_12(df: pd.DataFrame, embeds_folder:str) -> List[MDField]:
    fields = {}
    embeds = {}

    for idx, row in df.iterrows():
        if 'dag_step' in row.keys():
            block = DataStep(row['dag_step'], row['dag_descr'])
        else:
            block = None
        md_field = MDField(row['field_name'], row['field_description'], row['field_type'], block=block)
        fields.update({row['field_name']: md_field})

        embed = np.load(os.path.join(embeds_folder, f"{idx}.npy"))
        embeds.update({row['field_name']: embed})
    return fields, embeds

def field_converter_34(df: pd.DataFrame, embeds_folder:str) -> List[MDField]:
    fields = {}
    embeds = {}

    for idx, row in df.iterrows():
        block = DataStep(row['node_id'], row['node_descr'])
        md_field = MDField(row['field_name'], row['field_description'], row['field_type'], block=block)
        fields.update({row['field_name']: md_field})

        embed = np.load(os.path.join(embeds_folder, f"{idx}.npy"))
        embeds.update({row['field_name']: embed})
    return fields, embeds


In [8]:
convertors = {
    'm1': field_converter_12,
    'm2': field_converter_12,
    'm3': field_converter_34,
    'm4': field_converter_34,
}

for p_name, p_path in DF_PATHS.items():
    if p_name != 'm4':
        continue

    emd_path = EMBEDS_PATHS[p_name]
    cur_save_dir = os.path.join(STATS_DIR, p_name)
    os.makedirs(cur_save_dir, exist_ok=True)

    df = pd.read_csv(p_path, index_col=[0])

    for q_idx in range(QUERIES_CNT):
        queries_fields = df.loc[df['query_id'] == q_idx]

        md_name = queries_fields.iloc[0]['md_short_name']
        md_descr = queries_fields.iloc[0]['md_description']

        llm_judger = factory_func(md_name, md_descr)

        # collect per run stats
        for r in range(RUNS):
            if q_idx < 3:
                continue
            if q_idx == 3 and r == 0:
                continue

            df_qr_fields = queries_fields.loc[df['run_id'] == r]
            fields, embeds = convertors[p_name](df_qr_fields, emd_path)

            volume = len(fields.keys())
            repeated_part = count_self_repeating_part(fields, embeds, llm_judger, 0.95, 0.3, 1, 40)

            with open(os.path.join(cur_save_dir, f"q{q_idx}_r{r}_stat.json"), 'wt') as fo:
                json.dump({'q': q_idx, 'r': r, 'volume': volume, 'repeated_part': repeated_part}, fo)

Spend during last LLM call:  {'prompt': 4779, 'completion': 269, 'total': 5048}
[]
Spend during last LLM call:  {'prompt': 4818, 'completion': 140, 'total': 4958}
[]
Spend during last LLM call:  {'prompt': 4794, 'completion': 206, 'total': 5000}
[]
Spend during last LLM call:  {'prompt': 4764, 'completion': 184, 'total': 4948}
[]
Spend during last LLM call:  {'prompt': 4701, 'completion': 417, 'total': 5118}
[]
Spend during last LLM call:  {'prompt': 4683, 'completion': 268, 'total': 4951}
[]
Spend during last LLM call:  {'prompt': 4758, 'completion': 241, 'total': 4999}
[]
Spend during last LLM call:  {'prompt': 4793, 'completion': 269, 'total': 5062}
[]
Spend during last LLM call:  {'prompt': 4876, 'completion': 209, 'total': 5085}
[]
Spend during last LLM call:  {'prompt': 4843, 'completion': 214, 'total': 5057}
[]
Spend during last LLM call:  {'prompt': 4858, 'completion': 234, 'total': 5092}
[]
Spend during last LLM call:  {'prompt': 4928, 'completion': 336, 'total': 5264}
[]
Spen

In [ ]:


# You are a metadata field comparer for scientific datasets.

# You receive the user's query to generate dataset, metadata table name and description,
# also ONE TARGET field and a LIST OF CANDIDATE fields.
# Decide which candidates describe EXACTLY the same meaning, that a TARGET fields.

# COMPARE the fields using next RULES:

# 1. BLOCK MISMATCH --> the fields are NOT the same.
#   Sometimes TARGET and CANDIDATE fields carry a block label (like "[\"sample_preparation\" Data Workflow Step]"),
#   not only the name, description and type. If thier block labels are DIFFERENT, they CAN'T be the same!
#   Do not proceed the 2nd step in such situation.

# 2. SUBJECT MISMATCH --> NOT equivalent.
#   Identify the real-world subject of each field: the entity whose property is being recorded (e.g. the patient, the sample, the instrument,
#   the treatment, the experiment session, the file).
#   If the TARGET and a CANDIDATE describe a property of DIFFERENT subjects, they are NOT equivalent — even if their descriptions share vocabulary,
#   examples, or allowed values.
#   Skip Rule 3 for that candidate.


# 3. EXACTLY SAME PROPERTY AND MEASURMENT --> the fields ARE the same.
#   A candidate is equivalent if ALL of the following hold:
#     - No block conflict (no labels, or matching labels from Rule 1).
#     - The fields are applied to the same subject or property.
#     - Same real-world property in names AND units: the two fields describes the exactly property in provided research.
#     - Compatible data types in (float vs int is fine; string vs float is not).

#   These differences do NOT disqualify equivalence:
#     - Different field names ("temp_c" vs "temperature_celsius")
#     - Minor wording variation ("chamber temperature" vs "ambient temperature during imaging")
#     - Different units for the same quantity ("_celsius" vs "_kelvin")

#   If comparing fields have a different meanings in thier description - they are NOT the same (like voxel_size in images properties and in microscope). 
#   To find out that details, you can use your reasoning powers - don't hesitate to reason well.

# OUTPUT: return the valid json only with reasoning block and the names of fields that are equivalent to the target.
# If no candidate is equivalent, return an empty list.
# The example:
# {{
#     "reasoning": "your reasoning",
#     "same_fields_from_list": [*list of value(-s), or empty list*]
# }}

## Усреднение всех статов

In [7]:
all_p_stats = {
    'm1': './../data/valid_stats/m1',
    'm2': './../data/valid_stats/m2',
    'm3': './../data/valid_stats/m3',
    'm4': './../data/valid_stats/m4',
}

stats = []

for p_name, p_stats in all_p_stats.items():
    stats_files = os.listdir(p_stats)
    stats_files = [os.path.join(p_stats, fn) for fn in stats_files]

    cur_stats = []
    unique_stats = []
    for fn in stats_files:
        with open(fn, 'rt') as fo:
            stat_dict = json.load(fo)
            cur_stats.append(stat_dict['repeated_part'])
            unique_stats.append(stat_dict['volume'] * (1 - stat_dict['repeated_part']))

    stats.append({
        'p_name': p_name,
        'repeat_avg': np.mean(cur_stats),
        'repeat_std': np.std(cur_stats),
        'unique_avg': np.mean(unique_stats),
        'unique_std': np.std(unique_stats),
    })

In [8]:
stats_df = pd.DataFrame(stats)
stats_df.head(100)

,p_name,repeat_avg,repeat_std,unique_avg,unique_std
0,m1,0.021991,0.027330,58.083333,11.842426
1,m2,0.018626,0.027378,53.541667,15.718829
2,m3,0.060065,0.040382,133.416667,23.302211
3,m4,0.092683,0.068224,190.000000,59.550399
